## Teste com variáveis selecionadas - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança para garantir tipos numéricos
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 1: Variáveis Selecionadas (Sem Tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

print("=======================================================")
print("INTERLAGOS - CENÁRIO 1: OLS E RANDOM FOREST (SEM TEMPO)")
print("=======================================================\n")

# =====================================================================
# PARTE A: ESTATÍSTICA (OLS) para Teste F, Teste T e P-Valor
# =====================================================================
print("--- 1. RESULTADOS ESTATÍSTICOS (OLS) ---")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())
print(f"Prob (F-statistic): {model_ols.f_pvalue:.4e}")

# =====================================================================
# PARTE B: MACHINE LEARNING (RANDOM FOREST)
# =====================================================================
print("\n--- 2. RESULTADOS RANDOM FOREST ---")
rf_model = RandomForestRegressor(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
r2_rf = r2_score(y_test, y_pred)

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")

importances_rf = pd.Series(rf_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_rf * 100)

INTERLAGOS - CENÁRIO 1: OLS E RANDOM FOREST (SEM TEMPO)

--- 1. RESULTADOS ESTATÍSTICOS (OLS) ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.705
Model:                                                      OLS   Adj. R-squared:                  0.705
Method:                                           Least Squares   F-statistic:                     2792.
Date:                                          Tue, 23 Jun 2026   Prob (F-statistic):               0.00
Time:                                                  00:17:46   Log-Likelihood:                -16514.
No. Observations:                                          7008   AIC:                         3.304e+04
Df Residuals:                                              7001   BIC:                         3.309e+04
Df Model:                                                     

## Teste com todas as variáveis (Sem filtro) - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 2: TODAS as 11 Variáveis (Sem Tempo)
X_cols_all = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model = df[[target_col] + X_cols_all].copy()
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear').dropna()

X = df_model[X_cols_all]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

print("=======================================================")
print("INTERLAGOS - CENÁRIO 2: OLS E RANDOM FOREST (11 VARS - SEM TEMPO)")
print("=======================================================\n")

# =====================================================================
# PARTE A: ESTATÍSTICA (OLS)
# =====================================================================
print("--- 1. RESULTADOS ESTATÍSTICOS (OLS) ---")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())
print(f"Prob (F-statistic): {model_ols.f_pvalue:.4e}")

# =====================================================================
# PARTE B: MACHINE LEARNING (RANDOM FOREST)
# =====================================================================
print("\n--- 2. RESULTADOS RANDOM FOREST ---")
rf_model = RandomForestRegressor(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
r2_rf = r2_score(y_test, y_pred)

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")

importances_rf = pd.Series(rf_model.feature_importances_, index=X_cols_all).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_rf * 100)

INTERLAGOS - CENÁRIO 2: OLS E RANDOM FOREST (11 VARS - SEM TEMPO)

--- 1. RESULTADOS ESTATÍSTICOS (OLS) ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.709
Model:                                                      OLS   Adj. R-squared:                  0.708
Method:                                           Least Squares   F-statistic:                     1548.
Date:                                          Tue, 23 Jun 2026   Prob (F-statistic):               0.00
Time:                                                  00:18:21   Log-Likelihood:                -16472.
No. Observations:                                          7008   AIC:                         3.297e+04
Df Residuals:                                              6996   BIC:                         3.305e+04
Df Model:                                           

## Teste com variáveis selecionadas + Hora e mês - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo Hora e Mês)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 3. CENÁRIO 3: Variáveis Selecionadas + Hora e Mês
X_cols_time = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols_time].copy()
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear').dropna()

X = df_model[X_cols_time]
y = df_model[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

print("=======================================================")
print("INTERLAGOS - CENÁRIO 3: OLS E RANDOM FOREST (SELECIONADAS + TEMPO)")
print("=======================================================\n")

# =====================================================================
# PARTE A: ESTATÍSTICA (OLS)
# =====================================================================
print("--- 1. RESULTADOS ESTATÍSTICOS (OLS) ---")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())

# =====================================================================
# PARTE B: MACHINE LEARNING (RANDOM FOREST)
# =====================================================================
print("\n--- 2. RESULTADOS RANDOM FOREST ---")
rf_model = RandomForestRegressor(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
r2_rf = r2_score(y_test, y_pred)

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")

importances_rf = pd.Series(rf_model.feature_importances_, index=X_cols_time).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_rf * 100)

INTERLAGOS - CENÁRIO 3: OLS E RANDOM FOREST (SELECIONADAS + TEMPO)

--- 1. RESULTADOS ESTATÍSTICOS (OLS) ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.770
Model:                                                      OLS   Adj. R-squared:                  0.770
Method:                                           Least Squares   F-statistic:                     2933.
Date:                                          Tue, 23 Jun 2026   Prob (F-statistic):               0.00
Time:                                                  00:19:28   Log-Likelihood:                -15641.
No. Observations:                                          7008   AIC:                         3.130e+04
Df Residuals:                                              6999   BIC:                         3.136e+04
Df Model:                                          

## Teste com todas as variáveis (Sem filtro) + Hora e mês - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo Hora e Mês)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 3. CENÁRIO 4: TODAS as 11 Variáveis + Hora e Mês
X_cols_all_time = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols_all_time].copy()
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear').dropna()

X = df_model[X_cols_all_time]
y = df_model[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

print("=======================================================")
print("INTERLAGOS - CENÁRIO 4: OLS E RANDOM FOREST (TODAS + TEMPO)")
print("=======================================================\n")

# =====================================================================
# PARTE A: ESTATÍSTICA (OLS)
# =====================================================================
print("--- 1. RESULTADOS ESTATÍSTICOS (OLS) ---")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())

# =====================================================================
# PARTE B: MACHINE LEARNING (RANDOM FOREST)
# =====================================================================
print("\n--- 2. RESULTADOS RANDOM FOREST ---")
rf_model = RandomForestRegressor(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
r2_rf = r2_score(y_test, y_pred)

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")

importances_rf = pd.Series(rf_model.feature_importances_, index=X_cols_all_time).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_rf * 100)

INTERLAGOS - CENÁRIO 4: OLS E RANDOM FOREST (TODAS + TEMPO)

--- 1. RESULTADOS ESTATÍSTICOS (OLS) ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.773
Model:                                                      OLS   Adj. R-squared:                  0.773
Method:                                           Least Squares   F-statistic:                     1834.
Date:                                          Tue, 23 Jun 2026   Prob (F-statistic):               0.00
Time:                                                  00:20:01   Log-Likelihood:                -15596.
No. Observations:                                          7008   AIC:                         3.122e+04
Df Residuals:                                              6994   BIC:                         3.132e+04
Df Model:                                                 

## Teste com variáveis selecionadas - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 1: Variáveis Selecionadas (Sem Tempo)
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model_1 = df[[target_col] + X_cols_1].copy()
df_model_1['RADIACAO GLOBAL (Kj/m²)'] = df_model_1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_1 = df_model_1.interpolate(method='linear').dropna()

X_1 = df_model_1[X_cols_1]
y_1 = df_model_1[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model_1) * 0.8)
X_train_1, X_test_1 = X_1.iloc[:ponto_de_corte], X_1.iloc[ponto_de_corte:]
y_train_1, y_test_1 = y_1.iloc[:ponto_de_corte], y_1.iloc[ponto_de_corte:]

print("=======================================================")
print("INTERLAGOS - GRADIENT BOOSTING (CENÁRIO 1 - SEM TEMPO)")
print("=======================================================\n")

# 4. Treinando o Gradient Boosting
gb_1 = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_1.fit(X_train_1, y_train_1)

y_pred_1 = gb_1.predict(X_test_1)
r2_1 = r2_score(y_test_1, y_pred_1)

print(f"R² do Gradient Boosting em dados futuros: {r2_1:.4f}")

importances_1 = pd.Series(gb_1.feature_importances_, index=X_cols_1).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_1 * 100)

INTERLAGOS - GRADIENT BOOSTING (CENÁRIO 1 - SEM TEMPO)

R² do Gradient Boosting em dados futuros: 0.6652

Importância das Variáveis (em %):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    39.430531
UMIDADE RELATIVA DO AR, HORARIA (%)                      38.780136
RADIACAO GLOBAL (Kj/m²)                                  18.923432
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.598321
VENTO, VELOCIDADE HORARIA (m/s)                           1.171661
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.095919
dtype: float64


## Teste com todas as variáveis (Sem filtro) - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 2: TODAS as 11 Variáveis (Sem Tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df[[target_col] + X_cols_2].copy()
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model_2) * 0.8)
X_train_2, X_test_2 = X_2.iloc[:ponto_de_corte], X_2.iloc[ponto_de_corte:]
y_train_2, y_test_2 = y_2.iloc[:ponto_de_corte], y_2.iloc[ponto_de_corte:]

print("=======================================================")
print("INTERLAGOS - GRADIENT BOOSTING (CENÁRIO 2 - 11 VARS)")
print("=======================================================\n")

# 4. Treinando o Gradient Boosting
gb_2 = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_2.fit(X_train_2, y_train_2)

y_pred_2 = gb_2.predict(X_test_2)
r2_2 = r2_score(y_test_2, y_pred_2)

print(f"R² do Gradient Boosting em dados futuros: {r2_2:.4f}")

importances_2 = pd.Series(gb_2.feature_importances_, index=X_cols_2).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_2 * 100)

INTERLAGOS - GRADIENT BOOSTING (CENÁRIO 2 - 11 VARS)

R² do Gradient Boosting em dados futuros: 0.6760

Importância das Variáveis (em %):
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 25.796329
RADIACAO GLOBAL (Kj/m²)                                  19.082916
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         17.808398
UMIDADE RELATIVA DO AR, HORARIA (%)                      12.151144
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)          11.229300
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     9.380777
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.530490
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  1.448076
VENTO, RAJADA MAXIMA (m/s)                                0.856487
VENTO, VELOCIDADE HORARIA (m/s)                           0.666170
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.049913
dtype: float64


## Teste com variáveis selecionadas + Hora e mês - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo Hora e Mês)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 3. CENÁRIO 3: Variáveis Selecionadas + Hora e Mês
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df[[target_col] + X_cols_3].copy()
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model_3) * 0.8)
X_train_3, X_test_3 = X_3.iloc[:ponto_de_corte], X_3.iloc[ponto_de_corte:]
y_train_3, y_test_3 = y_3.iloc[:ponto_de_corte], y_3.iloc[ponto_de_corte:]

print("=======================================================")
print("INTERLAGOS - GRADIENT BOOSTING (CENÁRIO 3 - COM TEMPO)")
print("=======================================================\n")

# 5. Treinando o Gradient Boosting
gb_3 = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_3.fit(X_train_3, y_train_3)

y_pred_3 = gb_3.predict(X_test_3)
r2_3 = r2_score(y_test_3, y_pred_3)

print(f"R² do Gradient Boosting em dados futuros: {r2_3:.4f}")

importances_3 = pd.Series(gb_3.feature_importances_, index=X_cols_3).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_3 * 100)

INTERLAGOS - GRADIENT BOOSTING (CENÁRIO 3 - COM TEMPO)

R² do Gradient Boosting em dados futuros: 0.7620

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      34.958609
Mes                                                      31.989137
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    18.535451
RADIACAO GLOBAL (Kj/m²)                                  10.638822
Hora                                                      2.461503
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.802412
VENTO, VELOCIDADE HORARIA (m/s)                           0.613154
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.000912
dtype: float64


## Teste com todas as variáveis (Sem filtro) + Hora e mês - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo Hora e Mês)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 3. CENÁRIO 4: TODAS as 11 Variáveis + Hora e Mês
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df[[target_col] + X_cols_4].copy()
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model_4) * 0.8)
X_train_4, X_test_4 = X_4.iloc[:ponto_de_corte], X_4.iloc[ponto_de_corte:]
y_train_4, y_test_4 = y_4.iloc[:ponto_de_corte], y_4.iloc[ponto_de_corte:]

print("=======================================================")
print("INTERLAGOS - GRADIENT BOOSTING (CENÁRIO 4 - TODAS + TEMPO)")
print("=======================================================\n")

# 5. Treinando o Gradient Boosting
gb_4 = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_4.fit(X_train_4, y_train_4)

y_pred_4 = gb_4.predict(X_test_4)
r2_4 = r2_score(y_test_4, y_pred_4)

print(f"R² do Gradient Boosting em dados futuros: {r2_4:.4f}")

importances_4 = pd.Series(gb_4.feature_importances_, index=X_cols_4).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_4 * 100)

INTERLAGOS - GRADIENT BOOSTING (CENÁRIO 4 - TODAS + TEMPO)

R² do Gradient Boosting em dados futuros: 0.7690

Importância das Variáveis (em %):
Mes                                                      32.609533
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 25.166216
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    14.899317
RADIACAO GLOBAL (Kj/m²)                                  10.431959
UMIDADE RELATIVA DO AR, HORARIA (%)                       9.198352
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  2.567875
Hora                                                      1.821580
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          1.172537
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.803680
VENTO, RAJADA MAXIMA (m/s)                                0.466701
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           0.450764
VENTO, VELOCIDADE HORARIA (m/s)                           0.407635
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                    

## Teste com variáveis selecionadas - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_xgb1 = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df_xgb1.columns:
    if df_xgb1[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_xgb1[col] = pd.to_numeric(df_xgb1[col].str.replace(',', '.'), errors='coerce')
        except:
            df_xgb1[col] = pd.to_numeric(df_xgb1[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 1: Variáveis Selecionadas (Sem Tempo)
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model_1 = df_xgb1[[target_col] + X_cols_1].copy()
df_model_1['RADIACAO GLOBAL (Kj/m²)'] = df_model_1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_1 = df_model_1.interpolate(method='linear').dropna()

X_1 = df_model_1[X_cols_1]
y_1 = df_model_1[target_col]

# 3. Divisão Cronológica (80/20)
ponto_1 = int(len(df_model_1) * 0.8)
X_train_1, X_test_1 = X_1.iloc[:ponto_1], X_1.iloc[ponto_1:]
y_train_1, y_test_1 = y_1.iloc[:ponto_1], y_1.iloc[ponto_1:]

print("=======================================================")
print("MODELO: XGBOOST (VARIÁVEIS SELECIONADAS - SEM TEMPO)")
print("=======================================================\n")

# 4. Treinando o XGBoost Padrão
xgb_1 = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
xgb_1.fit(X_train_1, y_train_1)

y_pred_1 = xgb_1.predict(X_test_1)
r2_1 = r2_score(y_test_1, y_pred_1)

print(f"R² do XGBoost em dados futuros: {r2_1:.4f}")

importances_1 = pd.Series(xgb_1.feature_importances_, index=X_cols_1).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_1 * 100)

MODELO: XGBOOST (VARIÁVEIS SELECIONADAS - SEM TEMPO)

R² do XGBoost em dados futuros: 0.6631

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      51.411533
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    26.705780
RADIACAO GLOBAL (Kj/m²)                                  13.288042
VENTO, VELOCIDADE HORARIA (m/s)                           3.651488
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      3.492815
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          1.450339
dtype: float32


## Teste com todas as variáveis (Sem filtro) - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_xgb2 = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df_xgb2.columns:
    if df_xgb2[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_xgb2[col] = pd.to_numeric(df_xgb2[col].str.replace(',', '.'), errors='coerce')
        except:
            df_xgb2[col] = pd.to_numeric(df_xgb2[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 2: Todas as 11 Variáveis (Sem Tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df_xgb2[[target_col] + X_cols_2].copy()
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Divisão Cronológica (80/20)
ponto_2 = int(len(df_model_2) * 0.8)
X_train_2, X_test_2 = X_2.iloc[:ponto_2], X_2.iloc[ponto_2:]
y_train_2, y_test_2 = y_2.iloc[:ponto_2], y_2.iloc[ponto_2:]

print("=======================================================")
print("MODELO: XGBOOST (TODAS AS 11 VARIÁVEIS - SEM TEMPO)")
print("=======================================================\n")

# 4. Treinando o XGBoost Padrão
xgb_2 = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
xgb_2.fit(X_train_2, y_train_2)

y_pred_2 = xgb_2.predict(X_test_2)
r2_2 = r2_score(y_test_2, y_pred_2)

print(f"R² do XGBoost em dados futuros: {r2_2:.4f}")

importances_2 = pd.Series(xgb_2.feature_importances_, index=X_cols_2).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_2 * 100)

MODELO: XGBOOST (TODAS AS 11 VARIÁVEIS - SEM TEMPO)

R² do XGBoost em dados futuros: 0.6765

Importância das Variáveis (em %):
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 40.783218
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         23.081114
RADIACAO GLOBAL (Kj/m²)                                   7.602252
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           7.368287
UMIDADE RELATIVA DO AR, HORARIA (%)                       7.112799
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     5.259107
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  3.736625
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.685231
VENTO, RAJADA MAXIMA (m/s)                                1.373980
VENTO, VELOCIDADE HORARIA (m/s)                           1.368246
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.629134
dtype: float32


## Teste com variáveis selecionadas + Hora e mês - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_xgb3 = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df_xgb3.columns:
    if df_xgb3[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_xgb3[col] = pd.to_numeric(df_xgb3[col].str.replace(',', '.'), errors='coerce')
        except:
            df_xgb3[col] = pd.to_numeric(df_xgb3[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo Hora e Mês)
df_xgb3['Hora'] = df_xgb3['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_xgb3['Mes'] = pd.to_datetime(df_xgb3['Data']).dt.month

# 3. CENÁRIO 3: Variáveis Selecionadas + Hora e Mês
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df_xgb3[[target_col] + X_cols_3].copy()
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 4. Divisão Cronológica (80/20)
ponto_3 = int(len(df_model_3) * 0.8)
X_train_3, X_test_3 = X_3.iloc[:ponto_3], X_3.iloc[ponto_3:]
y_train_3, y_test_3 = y_3.iloc[:ponto_3], y_3.iloc[ponto_3:]

print("=======================================================")
print("MODELO: XGBOOST (VARIÁVEIS SELECIONADAS + HORA/MÊS)")
print("=======================================================\n")

# 5. Treinando o XGBoost Padrão
xgb_3 = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
xgb_3.fit(X_train_3, y_train_3)

y_pred_3 = xgb_3.predict(X_test_3)
r2_3 = r2_score(y_test_3, y_pred_3)

print(f"R² do XGBoost em dados futuros: {r2_3:.4f}")

importances_3 = pd.Series(xgb_3.feature_importances_, index=X_cols_3).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_3 * 100)

MODELO: XGBOOST (VARIÁVEIS SELECIONADAS + HORA/MÊS)

R² do XGBoost em dados futuros: 0.7656

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      38.887154
Mes                                                      37.186405
RADIACAO GLOBAL (Kj/m²)                                   9.230482
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     6.931124
Hora                                                      3.978470
VENTO, VELOCIDADE HORARIA (m/s)                           1.868504
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.463294
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.454575
dtype: float32


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_xgb4 = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df_xgb4.columns:
    if df_xgb4[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_xgb4[col] = pd.to_numeric(df_xgb4[col].str.replace(',', '.'), errors='coerce')
        except:
            df_xgb4[col] = pd.to_numeric(df_xgb4[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo Hora e Mês)
df_xgb4['Hora'] = df_xgb4['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_xgb4['Mes'] = pd.to_datetime(df_xgb4['Data']).dt.month

# 3. CENÁRIO 4: TODAS as 11 Variáveis + Hora e Mês
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df_xgb4[[target_col] + X_cols_4].copy()
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 4. Divisão Cronológica (80/20)
ponto_4 = int(len(df_model_4) * 0.8)
X_train_4, X_test_4 = X_4.iloc[:ponto_4], X_4.iloc[ponto_4:]
y_train_4, y_test_4 = y_4.iloc[:ponto_4], y_4.iloc[ponto_4:]

print("=======================================================")
print("MODELO: XGBOOST (TODAS AS VARIÁVEIS + HORA/MÊS)")
print("=======================================================\n")

# 5. Treinando o XGBoost Padrão
xgb_4 = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
xgb_4.fit(X_train_4, y_train_4)

y_pred_4 = xgb_4.predict(X_test_4)
r2_4 = r2_score(y_test_4, y_pred_4)

print(f"R² do XGBoost em dados futuros: {r2_4:.4f}")

importances_4 = pd.Series(xgb_4.feature_importances_, index=X_cols_4).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_4 * 100)

MODELO: XGBOOST (TODAS AS VARIÁVEIS + HORA/MÊS)

R² do XGBoost em dados futuros: 0.7649

Importância das Variáveis (em %):
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 42.091091
Mes                                                      24.567381
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  7.613551
UMIDADE RELATIVA DO AR, HORARIA (%)                       6.162398
RADIACAO GLOBAL (Kj/m²)                                   5.598186
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     3.403984
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          2.844996
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           2.361197
Hora                                                      2.007229
VENTO, VELOCIDADE HORARIA (m/s)                           1.026254
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.996225
VENTO, RAJADA MAXIMA (m/s)                                0.961728
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.365785
dtype:

## Teste com variáveis selecionadas - XGBoost tunado


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_tune1 = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df_tune1.columns:
    if df_tune1[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune1[col] = pd.to_numeric(df_tune1[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune1[col] = pd.to_numeric(df_tune1[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 1: Variáveis Selecionadas (Sem Tempo)
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model_1 = df_tune1[[target_col] + X_cols_1].copy()
df_model_1['RADIACAO GLOBAL (Kj/m²)'] = df_model_1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_1 = df_model_1.interpolate(method='linear').dropna()

X_1 = df_model_1[X_cols_1]
y_1 = df_model_1[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_1 = int(len(df_model_1) * 0.8)
X_train_1, X_test_1 = X_1.iloc[:ponto_1], X_1.iloc[ponto_1:]
y_train_1, y_test_1 = y_1.iloc[:ponto_1], y_1.iloc[ponto_1:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (INTERLAGOS - CENÁRIO 1)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros para Busca
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]
}

xgb_base_1 = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search_1 = RandomizedSearchCV(
    estimator=xgb_base_1,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Executando a Otimização
random_search_1.fit(X_train_1, y_train_1)
melhor_xgb_1 = random_search_1.best_estimator_

# 6. Avaliação do Modelo Otimizado
y_pred_1 = melhor_xgb_1.predict(X_test_1)
r2_1 = r2_score(y_test_1, y_pred_1)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 1) ---")
print(f"Melhores Hiperparâmetros: {random_search_1.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_1:.4f}")

importances_1 = pd.Series(melhor_xgb_1.feature_importances_, index=X_cols_1).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_1 * 100)

INICIANDO FINE TUNING: XGBOOST (INTERLAGOS - CENÁRIO 1)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 1) ---
Melhores Hiperparâmetros: {'subsample': 1.0, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.6}
R² do XGBoost Otimizado em dados futuros: 0.6545

Importância das Variáveis (em %):
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    40.042179
RADIACAO GLOBAL (Kj/m²)                                  32.101204
UMIDADE RELATIVA DO AR, HORARIA (%)                      18.795767
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      4.130086
VENTO, VELOCIDADE HORARIA (m/s)                           3.844925
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          1.085837
dtype: float32


## Teste com todas as variáveis (Sem filtro) - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_tune2 = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df_tune2.columns:
    if df_tune2[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune2[col] = pd.to_numeric(df_tune2[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune2[col] = pd.to_numeric(df_tune2[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 2: Todas as 11 Variáveis (Sem Tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df_tune2[[target_col] + X_cols_2].copy()
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Divisão Cronológica (80/20)
ponto_2 = int(len(df_model_2) * 0.8)
X_train_2, X_test_2 = X_2.iloc[:ponto_2], X_2.iloc[ponto_2:]
y_train_2, y_test_2 = y_2.iloc[:ponto_2], y_2.iloc[ponto_2:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (INTERLAGOS - CENÁRIO 2)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros para Busca
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]
}

xgb_base_2 = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search_2 = RandomizedSearchCV(
    estimator=xgb_base_2,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Executando a Otimização
random_search_2.fit(X_train_2, y_train_2)
melhor_xgb_2 = random_search_2.best_estimator_

# 6. Avaliação do Modelo Otimizado
y_pred_2 = melhor_xgb_2.predict(X_test_2)
r2_2 = r2_score(y_test_2, y_pred_2)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 2) ---")
print(f"Melhores Hiperparâmetros: {random_search_2.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_2:.4f}")

importances_2 = pd.Series(melhor_xgb_2.feature_importances_, index=X_cols_2).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_2 * 100)

INICIANDO FINE TUNING: XGBOOST (INTERLAGOS - CENÁRIO 2)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 2) ---
Melhores Hiperparâmetros: {'subsample': 0.6, 'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
R² do XGBoost Otimizado em dados futuros: 0.6793

Importância das Variáveis (em %):
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 25.574436
RADIACAO GLOBAL (Kj/m²)                                  15.704385
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         14.687671
UMIDADE RELATIVA DO AR, HORARIA (%)                      13.797683
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)          13.773786
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     9.186504
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  2.196238
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.704750
VENTO, RAJADA MAXIMA (m/s)     

## Teste com variáveis selecionadas + Hora e mês - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_tune3 = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df_tune3.columns:
    if df_tune3[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune3[col] = pd.to_numeric(df_tune3[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune3[col] = pd.to_numeric(df_tune3[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Hora e Mês)
df_tune3['Hora'] = df_tune3['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_tune3['Mes'] = pd.to_datetime(df_tune3['Data']).dt.month

# 3. CENÁRIO 3: Variáveis Selecionadas + Hora e Mês
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df_tune3[[target_col] + X_cols_3].copy()
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_3 = int(len(df_model_3) * 0.8)
X_train_3, X_test_3 = X_3.iloc[:ponto_3], X_3.iloc[ponto_3:]
y_train_3, y_test_3 = y_3.iloc[:ponto_3], y_3.iloc[ponto_3:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (INTERLAGOS - CENÁRIO 3)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 5. Definindo a Grade de Hiperparâmetros para Busca
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]
}

xgb_base_3 = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search_3 = RandomizedSearchCV(
    estimator=xgb_base_3,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 6. Executando a Otimização
random_search_3.fit(X_train_3, y_train_3)
melhor_xgb_3 = random_search_3.best_estimator_

# 7. Avaliação do Modelo Otimizado
y_pred_3 = melhor_xgb_3.predict(X_test_3)
r2_3 = r2_score(y_test_3, y_pred_3)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 3) ---")
print(f"Melhores Hiperparâmetros: {random_search_3.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_3:.4f}")

importances_3 = pd.Series(melhor_xgb_3.feature_importances_, index=X_cols_3).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_3 * 100)

INICIANDO FINE TUNING: XGBOOST (INTERLAGOS - CENÁRIO 3)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 3) ---
Melhores Hiperparâmetros: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
R² do XGBoost Otimizado em dados futuros: 0.7651

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      34.959209
Mes                                                      23.999361
RADIACAO GLOBAL (Kj/m²)                                  22.317932
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    11.641245
Hora                                                      3.777328
VENTO, VELOCIDADE HORARIA (m/s)                           1.514046
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.449399
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.341484
dtype: float32


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_tune4 = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df_tune4.columns:
    if df_tune4[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune4[col] = pd.to_numeric(df_tune4[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune4[col] = pd.to_numeric(df_tune4[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Hora e Mês)
df_tune4['Hora'] = df_tune4['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_tune4['Mes'] = pd.to_datetime(df_tune4['Data']).dt.month

# 3. CENÁRIO 4: TODAS as 11 Variáveis + Hora e Mês
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df_tune4[[target_col] + X_cols_4].copy()
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 4. Divisão Cronológica (80/20)
ponto_4 = int(len(df_model_4) * 0.8)
X_train_4, X_test_4 = X_4.iloc[:ponto_4], X_4.iloc[ponto_4:]
y_train_4, y_test_4 = y_4.iloc[:ponto_4], y_4.iloc[ponto_4:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (INTERLAGOS - CENÁRIO 4)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 5. Definindo a Grade de Hiperparâmetros para Busca
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]
}

xgb_base_4 = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search_4 = RandomizedSearchCV(
    estimator=xgb_base_4,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 6. Executando a Otimização
random_search_4.fit(X_train_4, y_train_4)
melhor_xgb_4 = random_search_4.best_estimator_

# 7. Avaliação do Modelo Otimizado
y_pred_4 = melhor_xgb_4.predict(X_test_4)
r2_4 = r2_score(y_test_4, y_pred_4)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 4) ---")
print(f"Melhores Hiperparâmetros: {random_search_4.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_4:.4f}")

importances_4 = pd.Series(melhor_xgb_4.feature_importances_, index=X_cols_4).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_4 * 100)

INICIANDO FINE TUNING: XGBOOST (INTERLAGOS - CENÁRIO 4)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 4) ---
Melhores Hiperparâmetros: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
R² do XGBoost Otimizado em dados futuros: 0.7602

Importância das Variáveis (em %):
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 29.596519
Mes                                                      16.256210
RADIACAO GLOBAL (Kj/m²)                                  13.673486
UMIDADE RELATIVA DO AR, HORARIA (%)                      11.949911
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          8.526017
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     6.164679
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  5.059968
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           3.588285
Hora                            

## Teste com variáveis selecionadas - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos (Mantendo o padrão rígido de arquivos)
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_interlagos = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança para garantir dados numéricos
for col in df_interlagos.columns:
    if df_interlagos[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col].str.replace(',', '.'), errors='coerce')
        except:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col], errors='coerce')

# 2. Definição das variáveis do Cenário 1
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

# Isolando as colunas e tratando valores nulos da Radiação
df_pca_int1 = df_interlagos[X_cols_1].copy()
df_pca_int1['RADIACAO GLOBAL (Kj/m²)'] = df_pca_int1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca_int1 = df_pca_int1.interpolate(method='linear').dropna()

# 3. Padronização Obrigatória (Z-score) via StandardScaler
scaler = StandardScaler()
X_scaled_int1 = scaler.fit_transform(df_pca_int1)

# 4. Executando o PCA para Interlagos
pca_int1 = PCA(random_state=42)
pca_int1.fit(X_scaled_int1)

# 5. Calculando a Variância Explicada
var_explicada_int1 = pca_int1.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: INTERLAGOS - CENÁRIO 1 (SEM TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada_int1[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada_int1[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada_int1[0] + var_explicada_int1[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) ordenado por relevância estrutural
loadings_int1 = pd.DataFrame(
    pca_int1.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_1))],
    index=X_cols_1
)

loadings_int1['Impacto_Absoluto_PC1'] = loadings_int1['PC1'].abs()
tabela_final_int1 = loadings_int1.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final_int1[['PC1', 'Impacto_Absoluto_PC1']])

PCA: INTERLAGOS - CENÁRIO 1 (SEM TEMPO)

Variância no Componente 1 (PC1): 33.63%
Variância no Componente 2 (PC2): 22.04%
Variância Acumulada (PC1 + PC2): 55.67%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.621694   
RADIACAO GLOBAL (Kj/m²)                            -0.617809   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.380975   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.220285   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.174270   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                    0.088151   

                                                    Impacto_Absoluto_PC1  
UMIDADE RELATIVA DO AR, HORARIA (%)                             0.621694  
RADIACAO GLOBAL (Kj/m²)                                         0.617809  
VENTO, VELOCIDADE HORARIA (m/s)                                 0.380975  
VENTO, DIREÇÃO HORARIA (gr)

## Teste com todas as variáveis (Sem filtro) - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos (Mantendo o padrão rígido de arquivos)
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_interlagos = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df_interlagos.columns:
    if df_interlagos[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col].str.replace(',', '.'), errors='coerce')
        except:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col], errors='coerce')

# 2. Definição das variáveis do Cenário 2 (Todas as 11 - Sem tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

# Isolando e tratando nulos
df_pca_int2 = df_interlagos[X_cols_2].copy()
df_pca_int2['RADIACAO GLOBAL (Kj/m²)'] = df_pca_int2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca_int2 = df_pca_int2.interpolate(method='linear').dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled_int2 = scaler.fit_transform(df_pca_int2)

# 4. Executando o PCA
pca_int2 = PCA(random_state=42)
pca_int2.fit(X_scaled_int2)

# 5. Calculando a Variância Explicada
var_explicada_int2 = pca_int2.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: INTERLAGOS - CENÁRIO 2 (11 VARS - SEM TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada_int2[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada_int2[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada_int2[0] + var_explicada_int2[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings_int2 = pd.DataFrame(
    pca_int2.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_2))],
    index=X_cols_2
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings_int2['Impacto_Absoluto_PC1'] = loadings_int2['PC1'].abs()
tabela_final_int2 = loadings_int2.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final_int2[['PC1', 'Impacto_Absoluto_PC1']])

PCA: INTERLAGOS - CENÁRIO 2 (11 VARS - SEM TEMPO)

Variância no Componente 1 (PC1): 37.58%
Variância no Componente 2 (PC2): 26.94%
Variância Acumulada (PC1 + PC2): 64.52%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)            0.441168   
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.436270   
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)            0.435712   
RADIACAO GLOBAL (Kj/m²)                            -0.368556   
VENTO, RAJADA MAXIMA (m/s)                         -0.250921   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.226905   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.225141   
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)    0.218871   
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)     0.215993   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.170655   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                    0.023331  

## Teste com variáveis selecionadas + Hora e mês - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos (Padrão rígido)
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_interlagos = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df_interlagos.columns:
    if df_interlagos[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col].str.replace(',', '.'), errors='coerce')
        except:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col], errors='coerce')

# Engenharia de Atributos para o Cenário 3 (Extraindo Hora e Mês)
df_interlagos['Hora'] = df_interlagos['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_interlagos['Mes'] = pd.to_datetime(df_interlagos['Data']).dt.month

# 2. Definição das variáveis do Cenário 3
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

# Isolando e tratando nulos
df_pca_int3 = df_interlagos[X_cols_3].copy()
df_pca_int3['RADIACAO GLOBAL (Kj/m²)'] = df_pca_int3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca_int3 = df_pca_int3.interpolate(method='linear').dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled_int3 = scaler.fit_transform(df_pca_int3)

# 4. Executando o PCA
pca_int3 = PCA(random_state=42)
pca_int3.fit(X_scaled_int3)

# 5. Calculando a Variância Explicada
var_explicada_int3 = pca_int3.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: INTERLAGOS - CENÁRIO 3 (SELECIONADAS + TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada_int3[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada_int3[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada_int3[0] + var_explicada_int3[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings_int3 = pd.DataFrame(
    pca_int3.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_3))],
    index=X_cols_3
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings_int3['Impacto_Absoluto_PC1'] = loadings_int3['PC1'].abs()
tabela_final_int3 = loadings_int3.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final_int3[['PC1', 'Impacto_Absoluto_PC1']])

PCA: INTERLAGOS - CENÁRIO 3 (SELECIONADAS + TEMPO)

Variância no Componente 1 (PC1): 29.11%
Variância no Componente 2 (PC2): 16.80%
Variância Acumulada (PC1 + PC2): 45.92%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
RADIACAO GLOBAL (Kj/m²)                             0.552392   
UMIDADE RELATIVA DO AR, HORARIA (%)                -0.546974   
Hora                                                0.436717   
VENTO, VELOCIDADE HORARIA (m/s)                     0.397848   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                0.150344   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI... -0.131600   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                   -0.062378   
Mes                                                 0.053532   

                                                    Impacto_Absoluto_PC1  
RADIACAO GLOBAL (Kj/m²)                                         0.552392  
UMIDADE RELATIVA DO AR, HORARIA (%)   

## Teste com todas as variáveis (Sem filtro) + Hora e mês - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos (Mantendo o padrão rígido de arquivos)
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_interlagos = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança para garantir dados numéricos
for col in df_interlagos.columns:
    if df_interlagos[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col].str.replace(',', '.'), errors='coerce')
        except:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col], errors='coerce')

# Engenharia de Atributos para o Cenário 4 (Extraindo Hora e Mês)
df_interlagos['Hora'] = df_interlagos['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_interlagos['Mes'] = pd.to_datetime(df_interlagos['Data']).dt.month

# 2. Definição das variáveis do Cenário 4 (Todas as 11 + Hora e Mês)
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

# Isolando as colunas e tratando valores nulos
df_pca_int4 = df_interlagos[X_cols_4].copy()
df_pca_int4['RADIACAO GLOBAL (Kj/m²)'] = df_pca_int4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca_int4 = df_pca_int4.interpolate(method='linear').dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled_int4 = scaler.fit_transform(df_pca_int4)

# 4. Executando o PCA
pca_int4 = PCA(random_state=42)
pca_int4.fit(X_scaled_int4)

# 5. Calculando a Variância Explicada
var_explicada_int4 = pca_int4.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: INTERLAGOS - CENÁRIO 4 (TODAS + TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada_int4[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada_int4[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada_int4[0] + var_explicada_int4[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings_int4 = pd.DataFrame(
    pca_int4.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_4))],
    index=X_cols_4
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings_int4['Impacto_Absoluto_PC1'] = loadings_int4['PC1'].abs()
tabela_final_int4 = loadings_int4.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final_int4[['PC1', 'Impacto_Absoluto_PC1']])

PCA: INTERLAGOS - CENÁRIO 4 (TODAS + TEMPO)

Variância no Componente 1 (PC1): 33.69%
Variância no Componente 2 (PC2): 23.04%
Variância Acumulada (PC1 + PC2): 56.73%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)            0.430582   
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.422878   
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)            0.422786   
RADIACAO GLOBAL (Kj/m²)                            -0.363682   
VENTO, RAJADA MAXIMA (m/s)                         -0.263198   
Hora                                               -0.260759   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.243136   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.191444   
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)    0.186093   
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)     0.182983   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.148348   
Mes 

## Teste com variáveis selecionadas - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_interlagos = pd.read_excel(nome_arquivo)

# Limpeza padrão de segurança para garantir dados numéricos
for col in df_interlagos.columns:
    if df_interlagos[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col].str.replace(',', '.'), errors='coerce')
        except:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 1 (Física pura, sem tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df_interlagos[[target_col] + X_cols].copy()
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Configurando o XGBoost com os Hiperparâmetros Otimizados do Tuning de Interlagos (C1)
modelo_xgb = XGBRegressor(
    subsample=0.8,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - INTERLAGOS C1")
print("Foco: Estabilidade Estrutural das Regras Físicas na Zona Sul")
print("Aviso: Risco de Data Leakage por interpolação de dados vizinhos")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_xgb, X, y, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - INTERLAGOS C1")
print("Foco: Capacidade Real de Previsão do Tempo (Próximo à Represa)")
print("Vantagem: Totalmente livre de Data Leakage (Proibido espiar o futuro)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    modelo_xgb.fit(X_train, y_train)
    score = modelo_xgb.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - INTERLAGOS C1
Foco: Estabilidade Estrutural das Regras Físicas na Zona Sul
Aviso: Risco de Data Leakage por interpolação de dados vizinhos
Dobra Aleatória 1: R² = 0.7739
Dobra Aleatória 2: R² = 0.7557
Dobra Aleatória 3: R² = 0.7528
Dobra Aleatória 4: R² = 0.7679
Dobra Aleatória 5: R² = 0.7549

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.7610
Desvio Padrão das Dobras: 0.0083

MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - INTERLAGOS C1
Foco: Capacidade Real de Previsão do Tempo (Próximo à Represa)
Vantagem: Totalmente livre de Data Leakage (Proibido espiar o futuro)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.1414
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = -0.6025
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.3063
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7037
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.6450

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global

## Teste com todas as variáveis (Sem filtro) - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_interlagos = pd.read_excel(nome_arquivo)

# Limpeza padrão de segurança para garantir dados numéricos
for col in df_interlagos.columns:
    if df_interlagos[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col].str.replace(',', '.'), errors='coerce')
        except:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 2 (Todas as 11 variáveis físicas, sem tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df_interlagos[[target_col] + X_cols_2].copy()
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Configurando o XGBoost com os Hiperparâmetros Otimizados do Tuning (C2)
modelo_xgb_2 = XGBRegressor(
    subsample=0.8,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - INTERLAGOS C2")
print("Foco: Estabilidade Estrutural com 11 Variáveis na Zona Sul")
print("Aviso: Risco de Data Leakage por interpolação de dados vizinhos")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_xgb_2, X_2, y_2, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - INTERLAGOS C2")
print("Foco: Capacidade Real de Previsão do Tempo (Sem Tempo, 11 Vars)")
print("Vantagem: Totalmente livre de Data Leakage (Proibido espiar o futuro)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_2)):
    X_train, X_test = X_2.iloc[train_index], X_2.iloc[test_index]
    y_train, y_test = y_2.iloc[train_index], y_2.iloc[test_index]

    modelo_xgb_2.fit(X_train, y_train)
    score = modelo_xgb_2.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - INTERLAGOS C2
Foco: Estabilidade Estrutural com 11 Variáveis na Zona Sul
Aviso: Risco de Data Leakage por interpolação de dados vizinhos
Dobra Aleatória 1: R² = 0.7813
Dobra Aleatória 2: R² = 0.7683
Dobra Aleatória 3: R² = 0.7619
Dobra Aleatória 4: R² = 0.7708
Dobra Aleatória 5: R² = 0.7631

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.7691
Desvio Padrão das Dobras: 0.0069

MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - INTERLAGOS C2
Foco: Capacidade Real de Previsão do Tempo (Sem Tempo, 11 Vars)
Vantagem: Totalmente livre de Data Leakage (Proibido espiar o futuro)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.1504
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = -0.7140
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.3172
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.6938
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.6522

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global 

## Teste com variáveis selecionadas + Hora e mês - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_interlagos = pd.read_excel(nome_arquivo)

# Limpeza padrão de segurança para garantir dados numéricos
for col in df_interlagos.columns:
    if df_interlagos[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col].str.replace(',', '.'), errors='coerce')
        except:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# Engenharia de Atributos para o Cenário 3 (Extraindo Hora e Mês)
df_interlagos['Hora'] = df_interlagos['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_interlagos['Mes'] = pd.to_datetime(df_interlagos['Data']).dt.month

# 2. Variáveis do Cenário 3 (Físicas Selecionadas + Tempo)
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df_interlagos[[target_col] + X_cols_3].copy()
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 3. Configurando o XGBoost com os Hiperparâmetros Otimizados do Tuning (C3)
modelo_xgb_3 = XGBRegressor(
    subsample=0.8,
    n_estimators=300,
    max_depth=5,
    learning_rate=0.03,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - INTERLAGOS C3")
print("Foco: Estabilidade Estrutural com inclusão de Hora e Mês")
print("Aviso: Alto risco de Data Leakage por interpolação temporal")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_xgb_3, X_3, y_3, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - INTERLAGOS C3")
print("Foco: Capacidade Real de Previsão do Tempo (Com Calendário)")
print("Vantagem: Livre de Data Leakage (Avaliando o impacto do tempo)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_3)):
    X_train, X_test = X_3.iloc[train_index], X_3.iloc[test_index]
    y_train, y_test = y_3.iloc[train_index], y_3.iloc[test_index]

    modelo_xgb_3.fit(X_train, y_train)
    score = modelo_xgb_3.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - INTERLAGOS C3
Foco: Estabilidade Estrutural com inclusão de Hora e Mês
Aviso: Alto risco de Data Leakage por interpolação temporal
Dobra Aleatória 1: R² = 0.9094
Dobra Aleatória 2: R² = 0.9050
Dobra Aleatória 3: R² = 0.8962
Dobra Aleatória 4: R² = 0.9008
Dobra Aleatória 5: R² = 0.8885

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.9000
Desvio Padrão das Dobras: 0.0072

MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - INTERLAGOS C3
Foco: Capacidade Real de Previsão do Tempo (Com Calendário)
Vantagem: Livre de Data Leakage (Avaliando o impacto do tempo)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.0209
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = 0.2462
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.6738
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7948
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.7495

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.4970
Desvi

## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost(Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Interlagos
nome_arquivo = 'Interlagos_Sao_Paulo_2025.xlsx'
df_interlagos = pd.read_excel(nome_arquivo)

# Limpeza padrão de segurança para garantir dados numéricos
for col in df_interlagos.columns:
    if df_interlagos[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col].str.replace(',', '.'), errors='coerce')
        except:
            df_interlagos[col] = pd.to_numeric(df_interlagos[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# Engenharia de Atributos para o Cenário 4 (Extraindo Hora e Mês)
df_interlagos['Hora'] = df_interlagos['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_interlagos['Mes'] = pd.to_datetime(df_interlagos['Data']).dt.month

# 2. Variáveis do Cenário 4 (Todas as 11 + Hora e Mês)
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df_interlagos[[target_col] + X_cols_4].copy()
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 3. Configurando o XGBoost com os Hiperparâmetros Otimizados do Tuning (C4)
modelo_xgb_4 = XGBRegressor(
    subsample=0.8,
    n_estimators=300,
    max_depth=5,
    learning_rate=0.03,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - INTERLAGOS C4")
print("Foco: Estabilidade Estrutural com 11 Variáveis + Tempo na Zona Sul")
print("Aviso: Alto risco de Data Leakage por interpolação temporal")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_xgb_4, X_4, y_4, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - INTERLAGOS C4")
print("Foco: Capacidade Real de Previsão do Tempo (Cenário de Estresse)")
print("Vantagem: Livre de Data Leakage (Todas as físicas colineares + Tempo)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_4)):
    X_train, X_test = X_4.iloc[train_index], X_4.iloc[test_index]
    y_train, y_test = y_4.iloc[train_index], y_4.iloc[test_index]

    modelo_xgb_4.fit(X_train, y_train)
    score = modelo_xgb_4.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - INTERLAGOS C4
Foco: Estabilidade Estrutural com 11 Variáveis + Tempo na Zona Sul
Aviso: Alto risco de Data Leakage por interpolação temporal
Dobra Aleatória 1: R² = 0.9148
Dobra Aleatória 2: R² = 0.9110
Dobra Aleatória 3: R² = 0.9004
Dobra Aleatória 4: R² = 0.9046
Dobra Aleatória 5: R² = 0.8968

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.9055
Desvio Padrão das Dobras: 0.0066

MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - INTERLAGOS C4
Foco: Capacidade Real de Previsão do Tempo (Cenário de Estresse)
Vantagem: Livre de Data Leakage (Todas as físicas colineares + Tempo)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.0890
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = 0.0856
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.6597
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7839
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.7416

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Glo